# 阶段 1 · 第 2 课：训练一个小神经网络

上一课你写好了能自动算梯度的 `Value` 引擎，存在 `micrograd.py` 里。这一课用它搭一个小神经网络，让它自己从 4 个例子里学会规律。

学完你会看到，大模型训练也只是同样的 4 步，重复几十万次：

1. **前向**：把输入喂进模型，得到预测
2. **算 loss**：预测和正确答案差多少
3. **反向**：`loss.backward()`，算出每个参数的梯度
4. **更新**：每个参数往让 loss 变小的方向挪一小步

**怎么上**：先确认第 1 课已经跑完（会生成 `micrograd.py`），然后从上往下运行。

In [1]:
import random
from pathlib import Path

import matplotlib.pyplot as plt

assert Path("micrograd.py").exists(), "先把第 1 课跑完：它会生成 micrograd.py"
from micrograd import Value

plt.rcParams["font.sans-serif"] = ["Arial Unicode MS"]  # 让图里能显示中文
random.seed(42)  # 固定随机数，保证每次初始参数一样，结果可复现

## 第 1 步 · 一个神经元

神经元做的事很简单：

1. 每个输入乘一个**权重** w，加起来，再加一个**偏置** b：`w1·x1 + w2·x2 + w3·x3 + b`
2. 把结果送进 **tanh**，压到 −1 到 1 之间

w 和 b 就是**参数**，一开始是随机数，训练就是在调它们。

**要用到的写法**：
- `zip(self.w, x)`：把两个列表一一配对，得到 (w1, x1)、(w2, x2)……
- `sum(一串数, 起始值)`：从起始值开始往上加。例如 `sum([1, 2, 3], 10)` 等于 16。
- 例子：`sum(wi * xi for wi, xi in zip(w, x))` 就是 w1·x1 + w2·x2 + …

<details><summary>提示</summary>

起始值放 `self.b`，就能把「加偏置」和「加权求和」一步做完。

</details>

<details><summary>答案</summary>

```python
act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
```

</details>

## 第 2 步 · 把神经元搭成网络

- **一层（Layer）**：好几个神经元，吃同样的输入，各自输出一个数
- **多层感知机（MLP）**：几层串起来，前一层的输出是后一层的输入

下面这格把三个类一起存成 `nn.py`（和第 1 课一样用 `%%writefile`），作业里会直接用。你只需要填 `Neuron` 里的 TODO 1，`Layer` 和 `MLP` 已经写好了。

In [4]:
%%writefile nn.py
import random

from micrograd import Value


class Neuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1, 1))

    def __call__(self, x):
        # TODO 1：算 w1·x1 + w2·x2 + … + b，存成 act
        act = sum(wi * xi for wi, xi in zip(self.w, x)) + self.b

        assert act is not None, "先完成 TODO 1"
        return act.tanh()

    def parameters(self):
        return self.w + [self.b]


class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]


class MLP:
    def __init__(self, nin, nouts):
        sizes = [nin] + nouts
        self.layers = [Layer(sizes[i], sizes[i + 1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

Overwriting nn.py


载入刚存好的 `nn.py`，试一个神经元，再建一个 `MLP(3, [4, 4, 1])`：3 个输入 → 4 个神经元 → 4 个神经元 → 1 个输出。改了上一格的话，两格都要重新运行。

In [5]:
import importlib

import nn

importlib.reload(nn)
from nn import MLP, Neuron

n = Neuron(3)
print("一个神经元的输出:", n([2.0, 3.0, -1.0]))

model = MLP(3, [4, 4, 1])
print("参数个数:", len(model.parameters()))

一个神经元的输出: Value(data=-0.9835, grad=0.0000)
参数个数: 41


**算一算**：为什么是 41 个参数？

<details><summary>提示</summary>

每个神经元有「输入个数」个 w，加 1 个 b。

</details>

<details><summary>答案</summary>

- 第 1 层：4 个神经元 × (3 个 w + 1 个 b) = 16
- 第 2 层：4 × (4 + 1) = 20
- 第 3 层：1 × (4 + 1) = 5

合计 41。GPT-3 是 1750 亿个，算法完全一样，只是多得多。

</details>

## 第 3 步 · 数据和 loss

4 个例子，每个有 3 个输入，正确答案是 1 或 −1。网络要学会：看到这 3 个数，输出对应的答案。

**loss** 衡量模型错了多少。这里用**平方误差**：每个例子 (预测 − 答案)²，全部加起来。全对时 loss 是 0，错得越多越大。

先看看还没训练的网络表现怎么样（直接运行）：

In [6]:
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0]  # 正确答案

ypred = [model(x) for x in xs]
loss = sum((yp - yt) ** 2 for yp, yt in zip(ypred, ys))
print("预测:", [round(yp.data, 3) for yp in ypred])
print("答案:", ys)
print("loss:", round(loss.data, 4))

预测: [-0.284, 0.191, -0.465, -0.272]
答案: [1.0, -1.0, -1.0, 1.0]
loss: 4.9728


## 第 4 步 · 训练循环

下面的循环把开头说的 4 步重复 100 次。你要填两处：

**TODO 2 · 清零梯度**：每一轮 `backward()` 之前，先把所有参数的 `grad` 设回 0。

**TODO 3 · 更新参数**：每个参数往让 loss 变小的方向挪一小步，步子大小由**学习率** `lr` 控制。

**要用到的写法**：
- `model.parameters()`：所有参数（Value 对象）的列表，用 for 循环逐个处理
- `p.data`：参数的值，`p.grad`：它的梯度
- `x -= 3` 等于 `x = x - 3`

TODO 2 为什么要清零，想一想上一课 `_backward` 里用的是 `+=` 还是 `=`：

<details><summary>提示 1</summary>

`_backward` 用的是 `+=`，梯度是往上**累加**的。

</details>

<details><summary>提示 2</summary>

不清零的话，这一轮算出的梯度会叠在上一轮的梯度上面。

</details>

<details><summary>答案</summary>

```python
for p in model.parameters():
    p.grad = 0.0
```

</details>

TODO 3 往哪个方向挪，想一想梯度的正负是什么意思：

<details><summary>提示 1</summary>

`p.grad > 0` 的意思是：p 变大，loss 也会变大。

</details>

<details><summary>提示 2</summary>

我们想让 loss 变小，所以 p 要往梯度的**反方向**走：梯度为正就减小 p，为负就增大 p。

</details>

<details><summary>答案</summary>

```python
for p in model.parameters():
    p.data -= lr * p.grad
```

这就是**梯度下降**。

</details>

In [7]:
random.seed(42)
model = MLP(3, [4, 4, 1])  # 重新建一个，从随机参数开始训练
lr = 0.05                  # 学习率：每步挪多远
losses = []

for step in range(100):
    # 1. 前向
    ypred = [model(x) for x in xs]
    # 2. 算 loss
    loss = sum((yp - yt) ** 2 for yp, yt in zip(ypred, ys))

    # TODO 2：把每个参数的梯度清零
    for p in model.parameters():
        p.grad = 0

    # 3. 反向
    loss.backward()

    # TODO 3：每个参数往梯度的反方向走一小步
    for p in model.parameters():
        p.data -= lr * p.grad

    losses.append(loss.data)
    if step % 10 == 0:
        print(f"第 {step:3d} 步  loss = {loss.data:.4f}")

assert losses[-1] < losses[0] * 0.5, "loss 几乎没降：检查 TODO 2 和 TODO 3"
print(f"\n最终 loss = {losses[-1]:.4f}")
print("预测:", [round(yp.data, 3) for yp in ypred])
print("答案:", ys)

第   0 步  loss = 5.2305
第  10 步  loss = 0.2131
第  20 步  loss = 0.0845
第  30 步  loss = 0.0502
第  40 步  loss = 0.0349
第  50 步  loss = 0.0265
第  60 步  loss = 0.0212
第  70 步  loss = 0.0176
第  80 步  loss = 0.0149
第  90 步  loss = 0.0130

最终 loss = 0.0116
预测: [0.945, -0.965, -0.93, 0.951]
答案: [1.0, -1.0, -1.0, 1.0]


In [ ]:
plt.figure(figsize=(7, 3.5))
plt.plot(losses)
plt.xlabel("训练步数")
plt.ylabel("loss")
plt.title("loss 一路下降，说明模型在学")
plt.grid(alpha=0.3)
plt.show()

## 跑完想一想

- 预测值接近 1 和 −1，但不完全等于。为什么？（提示：tanh 的输出范围）
- 把 `lr` 改成 `0.5`、`2.0` 再跑，loss 曲线会怎样？（作业中等题 2 会系统地做这个实验）

## 这和大模型有什么关系

| | 我们的小网络 | 大语言模型 |
|---|---|---|
| 参数 | 41 个 | 几十亿到上万亿个 |
| 输入 | 3 个数 | 一串文字（先变成数字，阶段 3 讲） |
| 正确答案 | 1 或 −1 | 下一个字是什么 |
| loss | 平方误差 | 交叉熵（阶段 2 讲） |
| 训练循环 | 前向 → loss → 反向 → 更新 | **完全一样** |

## 本课新词

| 词 | 意思 |
|---|---|
| 神经元 | 加权求和 + 偏置，再过一个激活函数 |
| 权重 / 偏置 | 神经元里的参数 w 和 b |
| 激活函数 | 加在加权求和后面的非线性函数，这里用 tanh |
| MLP | 多层感知机，几层神经元串起来 |
| loss | 衡量模型错了多少的数，训练就是让它变小 |
| 学习率（lr） | 每次更新参数时挪多远 |
| 梯度下降 | 参数往梯度反方向走一小步，反复做 |

作业在同目录的 `homework.ipynb`。